# DEC Clustering — Optimalisasi Silhouette Score
## Pipeline: HEIC Support → Grid Search (K & latent_dim) → DEC → Ground Truth Validation
---
**Tujuan Notebook**: Meningkatkan kualitas clustering gambar koleksi museum secara sistematis
dengan memperbaiki 5 kelemahan utama dari notebook DEC baseline:

| # | Masalah Baseline | Solusi di Sini |
|---|---|---|
| 1 | Format `.heic` tidak terbaca | `pillow-heif` + fallback PIL |
| 2 | `n_clusters=5` hardcoded salah | Grid search K + eksperimen K=10 |
| 3 | `latent_dim=64` heuristic | Grid search [32, 64, 128, 256] |
| 4 | AE loss = MSE saja | MSE + L2 normalization + Sparse L1 reg |
| 5 | Tidak ada ground truth validation | Confusion matrix vs nama folder museum |

> **Dataset**: 593 file `.heic` dari 10 kategori koleksi museum  
> **Environment**: Lokal Mac (no Google Colab dependency)


## Cell 1 — Install & Import Library

In [ ]:
# ============================================================
# CELL 1: INSTALL & IMPORT LIBRARY
# ============================================================

# Pastikan pillow-heif sudah terpasang untuk membaca file .heic
# Jalankan sekali jika belum:
# !pip install pillow-heif tensorflow-macos tensorflow-metal

# ============================================================
# FIX NUMPY COMPATIBILITY (wajib sebelum import tensorflow)
# numpy dari conda-forge menghapus beberapa alias lama yang
# masih dipakai oleh TensorFlow 2.16.x secara internal.
# ============================================================
import numpy as np
_np_compat_attrs = {
    "complex_" : np.complex128,
    "bool"     : np.bool_,
    "int"      : np.int_,
    "float"    : np.float64,
    "object"   : np.object_,
    "str"      : np.str_,
}
for _attr, _val in _np_compat_attrs.items():
    if not hasattr(np, _attr):
        setattr(np, _attr, _val)
del _np_compat_attrs, _attr, _val
print("[OK] numpy compatibility patch applied")

import os
import cv2
import json
import time
import shutil
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from collections import Counter

# Scikit-learn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    silhouette_samples,
    davies_bouldin_score,
    calinski_harabasz_score,
    confusion_matrix
)
from sklearn.preprocessing import LabelEncoder

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K, regularizers
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.optimizers import Adam

# pillow-heif untuk membaca format .heic
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    HEIF_SUPPORT = True
    print('[OK] pillow-heif berhasil diaktifkan — file .heic dapat dibaca')
except ImportError:
    HEIF_SUPPORT = False
    print('[WARNING] pillow-heif TIDAK terdeteksi!')
    print('         Install via: pip install pillow-heif')
    print('         File .heic tidak akan dapat diproses!')

warnings.filterwarnings('ignore')
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f'TensorFlow versi : {tf.__version__}')
print(f'GPU tersedia     : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Numpy versi      : {np.__version__}')


## Cell 2 — Konfigurasi Global

In [ ]:
# ============================================================
# CELL 2: KONFIGURASI GLOBAL
# ============================================================

CONFIG = {
    # --- PATH LOKAL ---
    'dataset_path'     : '/Users/mdaffaatstsaqif/Downloads/klaster asli',
    'output_root'      : '/Users/mdaffaatstsaqif/Downloads/klaster asli/hasil_optimized',

    # --- PREPROCESSING ---
    'img_size'         : 128,          # 128x128 untuk efisiensi lokal (dataset ~590 gambar)
    'valid_ext'        : ('.jpg', '.jpeg', '.png', '.bmp', '.heic'),

    # --- GRID SEARCH: JUMLAH CLUSTER ---
    'k_search_range'   : list(range(2, 16)),   # K = 2 hingga 15
    'k_fixed'          : 10,                   # eksperimen paralel dengan K=10 (sesuai folder museum)

    # --- GRID SEARCH: LATENT DIM ---
    'latent_search'    : [32, 64, 128, 256],   # dimensi laten yang akan diuji
    'latent_probe_epochs': 30,                 # epoch pendek untuk probe grid search

    # --- AUTOENCODER FINAL ---
    'ae_epochs'        : 100,
    'ae_batch_size'    : 32,
    'ae_lr'            : 1e-3,
    'ae_val_split'     : 0.15,
    'ae_patience'      : 15,

    # --- DEC ---
    'dec_epochs'       : 150,
    'dec_batch_size'   : 64,
    'dec_lr'           : 1e-4,
    'dec_lr_min'       : 1e-6,
    'update_interval'  : 5,
    'tol'              : 0.001,

    # --- KMEANS ---
    'kmeans_n_init'    : 50,           # banyak restart → stabilitas lebih tinggi
    'kmeans_max_iter'  : 1000,

    # --- REGULARISASI ---
    'sparse_l1'        : 1e-5,         # L1 activity regularization pada latent layer

    # --- VISUALISASI ---
    'tsne_perplexity'  : 30,
    'n_sample_vis'     : 6,
}

# Buat folder output
for subdir in ['grafik', 'model', 'cluster_dec', 'cluster_kmeans']:
    os.makedirs(os.path.join(CONFIG['output_root'], subdir), exist_ok=True)

print('Konfigurasi aktif:')
for k, v in CONFIG.items():
    print(f'  {k:<22}: {v}')


## Cell 3 — Fungsi Preprocessing (dengan HEIC Support)

In [ ]:
# ============================================================
# CELL 3: PREPROCESSING CITRA — HEIC + FORMAT STANDAR
# Pipeline: Load → Resize → Denoise → CLAHE → Sharpen → Normalize
# ============================================================

def load_image(image_path):
    """
    Baca gambar secara adaptif:
    - File .heic  : dibaca via pillow_heif (sudah register)
    - Format lain  : dibaca via OpenCV
    Selalu mengembalikan array NumPy RGB uint8.
    """
    ext = os.path.splitext(image_path)[1].lower()

    if ext == '.heic':
        if not HEIF_SUPPORT:
            raise ImportError('pillow-heif diperlukan untuk membaca .heic')
        pil_img = Image.open(image_path).convert('RGB')
        return np.array(pil_img, dtype=np.uint8)
    else:
        img = cv2.imread(image_path)
        if img is None:
            raise ValueError(f'Gagal membaca: {image_path}')
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def preprocess_image(image_path, img_size=128):
    """
    Pipeline preprocessing optimal untuk clustering:
    1. Load (adaptif HEIC/standar)
    2. Resize ke img_size x img_size
    3. Gaussian denoising ringan
    4. CLAHE pada kanal L (LAB color space)
    5. Unsharp masking ringan
    6. Normalize ke [0, 1]
    """
    try:
        img = load_image(image_path)                             # (H, W, 3) uint8 RGB
        img = cv2.resize(img, (img_size, img_size),
                         interpolation=cv2.INTER_AREA)          # Resize
        img = cv2.GaussianBlur(img, (3, 3), 0)                  # Denoise

        # CLAHE pada channel L
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        img = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

        # Unsharp masking
        blurred = cv2.GaussianBlur(img, (5, 5), 0)
        img = cv2.addWeighted(img, 1.5, blurred, -0.5, 0)
        img = np.clip(img, 0, 255).astype(np.uint8)

        return img.astype(np.float32) / 255.0                    # Normalize

    except Exception as e:
        print(f'[WARNING] Gagal proses {os.path.basename(image_path)}: {e}')
        return None


print('Fungsi preprocessing siap (HEIC + format standar).')


## Cell 4 — Load Dataset + Ekstraksi Ground Truth Label dari Nama Folder

In [ ]:
# ============================================================
# CELL 4: LOAD DATASET
# - Rekursif scan dataset_path
# - Ground truth label = nama folder (kategori museum)
# - Validasi setiap gambar
# ============================================================

def load_dataset(dataset_path, img_size=128, valid_ext=('.jpg', '.jpeg', '.png', '.bmp', '.heic')):
    """
    Scan rekursif dataset_path.
    Returns:
        images      : np.array (N, H, W, 3) float32
        image_paths : list of str
        gt_labels   : np.array (N,) int  — indeks kategori dari nama folder
        gt_names    : list of str — nama folder/kategori asli per gambar
        class_names : list of str — nama unik kategori (terurut)
    """
    all_paths   = []
    all_folders = []

    for root, _, files in os.walk(dataset_path):
        # Abaikan folder output agar tidak tercampur
        if 'hasil_optimized' in root or 'hasil_cluster' in root:
            continue
        folder_name = os.path.basename(root)
        for f in sorted(files):
            if f.lower().endswith(valid_ext) and not f.startswith('.'):
                all_paths.append(os.path.join(root, f))
                all_folders.append(folder_name)

    print(f'Total file ditemukan : {len(all_paths)}')

    # Encoding ground truth label
    le = LabelEncoder()
    le.fit(all_folders)
    class_names = list(le.classes_)
    print(f'Kategori ditemukan   : {len(class_names)}')
    for i, c in enumerate(class_names):
        count = all_folders.count(c)
        print(f'  [{i:02d}] {c:<30} — {count} gambar')

    images, paths_ok, folders_ok = [], [], []

    for path, folder in tqdm(zip(all_paths, all_folders),
                             total=len(all_paths), desc='Loading & Preprocessing'):
        img = preprocess_image(path, img_size)
        if img is not None:
            images.append(img)
            paths_ok.append(path)
            folders_ok.append(folder)

    images    = np.array(images, dtype='float32')
    gt_labels = le.transform(folders_ok).astype(int)
    gt_names  = folders_ok

    print(f'\nBerhasil dimuat  : {len(images)} gambar')
    print(f'Shape dataset    : {images.shape}')
    print(f'Min/Max pixel    : {images.min():.4f} / {images.max():.4f}')
    print(f'Distribusi ground truth:')
    for i, c in enumerate(class_names):
        n = np.sum(gt_labels == i)
        bar = '█' * (n // 3)
        print(f'  [{i:02d}] {c:<30} {n:>3} {bar}')

    return images, paths_ok, gt_labels, gt_names, class_names


images, image_paths, gt_labels, gt_names, class_names = load_dataset(
    CONFIG['dataset_path'],
    CONFIG['img_size'],
    CONFIG['valid_ext']
)

# Simpan paths
with open(os.path.join(CONFIG['output_root'], 'image_paths.json'), 'w') as f:
    json.dump({'paths': image_paths, 'gt_labels': gt_labels.tolist(),
               'gt_names': gt_names, 'class_names': class_names}, f, indent=2)

# Preview 10 sampel
n_prev = min(10, len(images))
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.flatten()
indices = np.random.choice(len(images), n_prev, replace=False)
for i, idx in enumerate(indices):
    axes[i].imshow(images[idx])
    axes[i].set_title(gt_names[idx], fontsize=8, pad=3)
    axes[i].axis('off')
plt.suptitle('Preview Dataset (10 sampel acak)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '00_preview_dataset.png'),
            dpi=150, bbox_inches='tight')
plt.show()


## Cell 5 — Fase A: Eksplorasi K Optimal (Silhouette Score + Elbow)
> Sebelum melatih DEC penuh, kita cari K terbaik menggunakan K-Means cepat
> pada fitur gambar yang sudah di-flatten + PCA 50D.

In [ ]:
# ============================================================
# CELL 5: EKSPLORASI K OPTIMAL
# Menggunakan K-Means pada PCA-50D dari flat image pixels
# untuk estimasi awal sebelum latent space terbentuk
# ============================================================

print('Menjalankan eksplorasi K optimal...')
print('(Menggunakan PCA-50D dari pixel flatten sebagai proxy latent space awal)')

# Flatten images untuk PCA
images_flat = images.reshape(len(images), -1)

# PCA 50 komponen untuk mempercepat K-Means
n_pca = min(50, images_flat.shape[1], len(images) - 1)
pca_probe = PCA(n_components=n_pca, random_state=SEED)
X_pca = pca_probe.fit_transform(images_flat)
print(f'PCA-{n_pca}D: variance explained = {pca_probe.explained_variance_ratio_.sum()*100:.1f}%')

sil_scores  = []
inertias    = []
k_range     = CONFIG['k_search_range']

for k in tqdm(k_range, desc='K search'):
    km = KMeans(
        n_clusters=k,
        n_init=10,           # cepat untuk probe
        max_iter=300,
        random_state=SEED,
        init='k-means++'
    )
    labels_k = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels_k)
    sil_scores.append(sil)
    inertias.append(km.inertia_)
    print(f'  K={k:2d} | Silhouette={sil:.4f} | Inertia={km.inertia_:.1f}')

# Tentukan K optimal (Silhouette tertinggi)
best_k_idx   = int(np.argmax(sil_scores))
K_OPTIMAL    = k_range[best_k_idx]
print(f'\nK Optimal (Silhouette tertinggi) : K = {K_OPTIMAL}')
print(f'Silhouette Score pada K optimal  : {sil_scores[best_k_idx]:.4f}')

# Plot Silhouette + Elbow berdampingan
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.plot(k_range, sil_scores, 'o-', color='darkorange', linewidth=2, markersize=7)
ax.axvline(K_OPTIMAL, color='red', linestyle='--', linewidth=1.5,
           label=f'K optimal = {K_OPTIMAL}')
ax.axvline(CONFIG['k_fixed'], color='blue', linestyle=':', linewidth=1.5,
           label=f'K fixed (museum) = {CONFIG["k_fixed"]}')
ax.set_xlabel('Jumlah Cluster (K)', fontsize=12)
ax.set_ylabel('Silhouette Score', fontsize=12)
ax.set_title('Silhouette Score vs K\n(PCA-50D Proxy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)

ax = axes[1]
ax.plot(k_range, inertias, 's-', color='steelblue', linewidth=2, markersize=7)
ax.axvline(K_OPTIMAL, color='red', linestyle='--', linewidth=1.5, label=f'K optimal = {K_OPTIMAL}')
ax.set_xlabel('Jumlah Cluster (K)', fontsize=12)
ax.set_ylabel('Inertia (Within-cluster SSE)', fontsize=12)
ax.set_title('Elbow Method\n(PCA-50D Proxy)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Fase A: Eksplorasi K Optimal', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '01_k_search.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Simpan hasil
k_search_results = {'k_range': k_range, 'silhouette': sil_scores, 'inertia': inertias,
                    'k_optimal': K_OPTIMAL, 'k_fixed': CONFIG['k_fixed']}
with open(os.path.join(CONFIG['output_root'], 'k_search_results.json'), 'w') as f:
    json.dump(k_search_results, f, indent=2)


## Cell 6 — Fase B: Eksplorasi Latent Dim Optimal (Grid Search)
> Melatih Autoencoder ringan (~30 epoch) untuk setiap kandidat latent_dim,
> lalu memilih yang menghasilkan Silhouette Score tertinggi.

In [ ]:
# ============================================================
# CELL 6: GRID SEARCH LATENT DIM
# Training singkat (probe) untuk tiap latent_dim kandidat
# Menggunakan K_OPTIMAL yang sudah ditemukan
# ============================================================

def build_probe_autoencoder(input_shape, latent_dim, sparse_l1=0.0):
    """Autoencoder ringan untuk probe grid search."""
    H, W, C = input_shape
    inp = layers.Input(shape=input_shape)

    # Encoder (versi lebih ringan: 3 blok)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D(2)(x)          # /2
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)          # /4
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(2)(x)          # /8
    shape_bf = K.int_shape(x)[1:]
    x = layers.Flatten()(x)
    x = layers.Dense(256, activation='relu')(x)
    # Latent layer dengan L2 normalization + optional sparse reg
    latent_raw  = layers.Dense(latent_dim)(x)
    if sparse_l1 > 0:
        latent_raw = layers.ActivityRegularization(l1=sparse_l1)(latent_raw)
    latent = layers.Lambda(
        lambda z: tf.math.l2_normalize(z, axis=1),
        name='latent_l2'
    )(latent_raw)
    encoder_probe = models.Model(inp, latent, name=f'enc_probe_{latent_dim}')

    # Decoder
    n_flat = shape_bf[0] * shape_bf[1] * shape_bf[2]
    d_in = layers.Input(shape=(latent_dim,))
    d = layers.Dense(256, activation='relu')(d_in)
    d = layers.Dense(n_flat, activation='relu')(d)
    d = layers.Reshape(shape_bf)(d)
    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(128, 3, activation='relu', padding='same')(d)
    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(64, 3, activation='relu', padding='same')(d)
    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(32, 3, activation='relu', padding='same')(d)
    d_out = layers.Conv2D(C, 3, activation='sigmoid', padding='same')(d)
    decoder_probe = models.Model(d_in, d_out, name=f'dec_probe_{latent_dim}')

    ae_in  = layers.Input(shape=input_shape)
    ae_out = decoder_probe(encoder_probe(ae_in))
    ae = models.Model(ae_in, ae_out, name=f'ae_probe_{latent_dim}')
    ae.compile(optimizer=Adam(1e-3), loss='mse')
    return ae, encoder_probe


print('Memulai Grid Search Latent Dim...')
print(f'Kandidat: {CONFIG["latent_search"]}')
print(f'K yang digunakan: K_OPTIMAL={K_OPTIMAL}, K_FIXED={CONFIG["k_fixed"]}')
print('-' * 60)

ld_results = []   # list of dict

input_shape = (CONFIG['img_size'], CONFIG['img_size'], 3)

for ld in CONFIG['latent_search']:
    print(f'\n[latent_dim={ld}] Training probe {CONFIG["latent_probe_epochs"]} epoch...')
    ae_p, enc_p = build_probe_autoencoder(input_shape, ld, CONFIG['sparse_l1'])

    ae_p.fit(
        images, images,
        epochs=CONFIG['latent_probe_epochs'],
        batch_size=CONFIG['ae_batch_size'],
        validation_split=0.15,
        shuffle=True,
        verbose=0,
        callbacks=[EarlyStopping(monitor='val_loss', patience=5,
                                  restore_best_weights=True)]
    )

    feat = enc_p.predict(images, verbose=0)

    # Evaluasi dengan K_OPTIMAL
    km_o = KMeans(n_clusters=K_OPTIMAL, n_init=20, random_state=SEED, init='k-means++')
    lbl_o = km_o.fit_predict(feat)
    sil_o = silhouette_score(feat, lbl_o) if len(np.unique(lbl_o)) > 1 else -1

    # Evaluasi dengan K_FIXED=10
    km_f = KMeans(n_clusters=CONFIG['k_fixed'], n_init=20, random_state=SEED, init='k-means++')
    lbl_f = km_f.fit_predict(feat)
    sil_f = silhouette_score(feat, lbl_f) if len(np.unique(lbl_f)) > 1 else -1

    ld_results.append({
        'latent_dim'      : ld,
        'sil_k_optimal'   : round(float(sil_o), 4),
        'sil_k_fixed_10'  : round(float(sil_f), 4),
    })
    print(f'  Silhouette (K={K_OPTIMAL}): {sil_o:.4f}  |  Silhouette (K=10): {sil_f:.4f}')

    # Bersihkan memori
    del ae_p, enc_p, feat
    tf.keras.backend.clear_session()

# Pilih latent_dim terbaik (berdasarkan K_OPTIMAL)
best_ld_idx   = int(np.argmax([r['sil_k_optimal'] for r in ld_results]))
LATENT_DIM    = ld_results[best_ld_idx]['latent_dim']

print('\n' + '='*60)
print('HASIL GRID SEARCH LATENT DIM')
print('='*60)
print(f'{"latent_dim":<12} {"Sil(K_opt)":>12} {"Sil(K=10)":>12}')
print('-'*40)
for r in ld_results:
    flag = ' ← TERPILIH' if r['latent_dim'] == LATENT_DIM else ''
    print(f'{r["latent_dim"]:<12} {r["sil_k_optimal"]:>12.4f} {r["sil_k_fixed_10"]:>12.4f}{flag}')
print(f'\nLatent Dim Optimal: {LATENT_DIM}')

# Simpan hasil
with open(os.path.join(CONFIG['output_root'], 'latent_dim_search.json'), 'w') as f:
    json.dump({'results': ld_results, 'best_latent_dim': LATENT_DIM,
               'K_optimal': K_OPTIMAL, 'K_fixed': CONFIG['k_fixed']}, f, indent=2)

# Plot perbandingan
ld_labels  = [str(r['latent_dim']) for r in ld_results]
sil_o_vals = [r['sil_k_optimal'] for r in ld_results]
sil_f_vals = [r['sil_k_fixed_10'] for r in ld_results]
x = np.arange(len(ld_labels))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - w/2, sil_o_vals, w, label=f'K={K_OPTIMAL} (optimal)', color='darkorange')
bars2 = ax.bar(x + w/2, sil_f_vals, w, label=f'K={CONFIG["k_fixed"]} (fixed museum)', color='steelblue')
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([f'ld={l}' for l in ld_labels])
ax.set_xlabel('Latent Dimension', fontsize=12)
ax.set_ylabel('Silhouette Score (Probe)', fontsize=12)
ax.set_title('Fase B: Grid Search Latent Dim', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '02_latent_dim_search.png'),
            dpi=200, bbox_inches='tight')
plt.show()


## Cell 7 — Arsitektur Autoencoder Final
> CNN Autoencoder dengan L2 normalization dan Sparse L1 regularization pada latent space
> menggunakan `LATENT_DIM` optimal yang sudah ditemukan di Cell 6.

In [ ]:
# ============================================================
# CELL 7: ARSITEKTUR CNN AUTOENCODER FINAL (OPTIMIZED)
# Peningkatan vs baseline:
# 1. L2 normalization pada latent layer (hypersphere embedding)
# 2. Sparse L1 activity regularization (mendorong sparsity)
# 3. BatchNormalization lebih konsisten
# ============================================================

def build_autoencoder_optimized(input_shape=(128, 128, 3),
                                latent_dim=64,
                                sparse_l1=1e-5):
    """
    CNN Autoencoder dengan:
    - Encoder 4-blok Conv+BN+MaxPool → Dense → latent (L2-normalized)
    - Decoder 4-blok Dense+UpSample+Conv+BN → Conv output
    Returns: autoencoder, encoder, decoder
    """
    H, W, C = input_shape

    # ---- ENCODER ----
    enc_in = layers.Input(shape=input_shape, name='enc_input')
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(enc_in)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)         # H/2

    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)         # H/4

    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)         # H/8

    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, padding='same')(x)         # H/16

    shape_bf = K.int_shape(x)[1:]
    x = layers.Flatten()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Latent space dengan L2 normalization + sparse regularization
    latent_raw  = layers.Dense(latent_dim, name='latent_raw')(x)
    if sparse_l1 > 0:
        latent_raw = layers.ActivityRegularization(
            l1=sparse_l1, name='sparse_reg'
        )(latent_raw)
    latent = layers.Lambda(
        lambda z: tf.math.l2_normalize(z, axis=1),
        name='latent_l2norm'
    )(latent_raw)

    encoder = models.Model(enc_in, latent, name='Encoder_Optimized')

    # ---- DECODER ----
    n_flat = shape_bf[0] * shape_bf[1] * shape_bf[2]
    dec_in = layers.Input(shape=(latent_dim,), name='dec_input')
    d = layers.Dense(512, activation='relu')(dec_in)
    d = layers.Dense(n_flat, activation='relu')(d)
    d = layers.Reshape(shape_bf)(d)

    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(256, 3, activation='relu', padding='same')(d)
    d = layers.BatchNormalization()(d)

    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(128, 3, activation='relu', padding='same')(d)
    d = layers.BatchNormalization()(d)

    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(64, 3, activation='relu', padding='same')(d)
    d = layers.BatchNormalization()(d)

    d = layers.UpSampling2D(2)(d)
    d = layers.Conv2D(32, 3, activation='relu', padding='same')(d)
    d_out = layers.Conv2D(C, 3, activation='sigmoid', padding='same', name='dec_output')(d)

    decoder = models.Model(dec_in, d_out, name='Decoder_Optimized')

    # ---- AUTOENCODER ----
    ae_in  = layers.Input(shape=input_shape, name='ae_input')
    encoded = encoder(ae_in)
    decoded = decoder(encoded)
    autoencoder = models.Model(ae_in, decoded, name='Autoencoder_Optimized')

    return autoencoder, encoder, decoder


autoencoder, encoder, decoder = build_autoencoder_optimized(
    input_shape=(CONFIG['img_size'], CONFIG['img_size'], 3),
    latent_dim=LATENT_DIM,
    sparse_l1=CONFIG['sparse_l1']
)

autoencoder.compile(optimizer=Adam(CONFIG['ae_lr']), loss='mse')

print(f'Autoencoder siap. Latent dim = {LATENT_DIM} (L2-normalized + Sparse L1)')
print(f'Parameter encoder: {encoder.count_params():,}')
print(f'Parameter total  : {autoencoder.count_params():,}')
encoder.summary()


## Cell 8 — Training Autoencoder Full

In [ ]:
# ============================================================
# CELL 8: TRAINING AUTOENCODER FULL (OPTIMIZED)
# Callbacks: EarlyStopping + ReduceLROnPlateau + ModelCheckpoint
# ============================================================

model_dir = os.path.join(CONFIG['output_root'], 'model')
os.makedirs(model_dir, exist_ok=True)

callbacks_ae = [
    EarlyStopping(
        monitor='val_loss', patience=CONFIG['ae_patience'],
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7,
        min_lr=1e-6, verbose=1
    ),
    ModelCheckpoint(
        filepath=os.path.join(model_dir, 'ae_best.keras'),
        monitor='val_loss', save_best_only=True, verbose=1
    )
]

print('Mulai training Autoencoder...')
print(f'  Epochs    : {CONFIG["ae_epochs"]}')
print(f'  Batch     : {CONFIG["ae_batch_size"]}')
print(f'  Val Split : {CONFIG["ae_val_split"]}')
print(f'  Latent    : {LATENT_DIM}D (L2-normalized)')
print('-' * 50)

t0 = time.time()
history_ae = autoencoder.fit(
    images, images,
    epochs=CONFIG['ae_epochs'],
    batch_size=CONFIG['ae_batch_size'],
    validation_split=CONFIG['ae_val_split'],
    shuffle=True,
    callbacks=callbacks_ae,
    verbose=1
)
elapsed_ae = time.time() - t0

print(f'\nTraining selesai: {elapsed_ae/60:.1f} menit')
print(f'Best val_loss   : {min(history_ae.history["val_loss"]):.6f}')

# Simpan model
autoencoder.save(os.path.join(model_dir, 'autoencoder_final.keras'))
encoder.save(os.path.join(model_dir, 'encoder_final.keras'))
decoder.save(os.path.join(model_dir, 'decoder_final.keras'))
print('Model tersimpan.')


## Cell 9 — Grafik Training Autoencoder

In [ ]:
# ============================================================
# CELL 9: GRAFIK TRAINING AUTOENCODER
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history_ae.history['loss']) + 1)

ax = axes[0]
ax.plot(epochs_ran, history_ae.history['loss'], 'b-o', ms=3, lw=2, label='Train Loss')
ax.plot(epochs_ran, history_ae.history['val_loss'], 'r-s', ms=3, lw=2, label='Val Loss')
ax.set_title('Loss Curve Autoencoder', fontsize=13, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)

# Contoh rekonstruksi
ax = axes[1]
n_show = 5
idx_show = np.random.choice(len(images), n_show, replace=False)
orig = images[idx_show]
recon = autoencoder.predict(orig, verbose=0)
mosaic = np.concatenate([np.concatenate(orig, axis=1),
                         np.concatenate(recon, axis=1)], axis=0)
ax.imshow(np.clip(mosaic, 0, 1))
ax.set_title(f'Rekonstruksi (atas=asli, bawah=rekonstruksi)\nlatent_dim={LATENT_DIM}',
             fontsize=11, fontweight='bold')
ax.axis('off')

plt.suptitle('Autoencoder: Training & Rekonstruksi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '03_ae_training.png'),
            dpi=200, bbox_inches='tight')
plt.show()


## Cell 10 — Ekstraksi Fitur Laten + Normalisasi

In [ ]:
# ============================================================
# CELL 10: EKSTRAKSI LATENT FEATURES
# ============================================================

print('Mengekstraksi latent features...')
latent_features = encoder.predict(images, batch_size=CONFIG['ae_batch_size'], verbose=1)

print(f'Shape latent features : {latent_features.shape}')
print(f'Min / Max             : {latent_features.min():.4f} / {latent_features.max():.4f}')
print(f'Mean / Std            : {latent_features.mean():.4f} / {latent_features.std():.4f}')

# Verifikasi L2 norm (semua ~1.0 karena L2-normalized)
norms = np.linalg.norm(latent_features, axis=1)
print(f'L2 norm (mean/std)    : {norms.mean():.4f} / {norms.std():.6f}')
print('(Jika mendekati 1.0/0.0 = L2 normalization berfungsi dengan benar)')

np.save(os.path.join(CONFIG['output_root'], 'latent_features.npy'), latent_features)
print('Latent features tersimpan.')


## Cell 11 — K-Means Final (n_init=50, k-means++)

In [ ]:
# ============================================================
# CELL 11: K-MEANS FINAL — INISIALISASI STABIL
# n_init=50: 50 restart untuk menghindari local minimum
# init='k-means++': inisialisasi cerdas yang menyebar
# ============================================================

def run_kmeans_stable(features, n_clusters, n_init=50, seed=42):
    """Menjalankan K-Means dengan banyak restart untuk stabilitas maksimal."""
    print(f'Menjalankan K-Means (K={n_clusters}, n_init={n_init})...')
    km = KMeans(
        n_clusters=n_clusters,
        n_init=n_init,
        max_iter=CONFIG['kmeans_max_iter'],
        init='k-means++',
        random_state=seed
    )
    labels = km.fit_predict(features)
    centers = km.cluster_centers_
    sil = silhouette_score(features, labels)
    dbi = davies_bouldin_score(features, labels)
    chi = calinski_harabasz_score(features, labels)
    print(f'  Silhouette Score : {sil:.4f}')
    print(f'  Davies-Bouldin   : {dbi:.4f}')
    print(f'  Calinski-Harabasz: {chi:.1f}')
    return km, labels, centers, sil, dbi, chi


print('=== K-Means dengan K OPTIMAL ===')
km_opt, km_labels_opt, km_centers_opt, km_sil_opt, km_dbi_opt, km_chi_opt = \
    run_kmeans_stable(latent_features, K_OPTIMAL,
                      n_init=CONFIG['kmeans_n_init'], seed=SEED)

print('\n=== K-Means dengan K FIXED=10 (museum) ===')
km_fix, km_labels_fix, km_centers_fix, km_sil_fix, km_dbi_fix, km_chi_fix = \
    run_kmeans_stable(latent_features, CONFIG['k_fixed'],
                      n_init=CONFIG['kmeans_n_init'], seed=SEED)

# Label yang dipakai untuk DEC (gunakan K_FIXED=10 sesuai arahan)
N_CLUSTERS    = CONFIG['k_fixed']
kmeans_labels = km_labels_fix
kmeans_centers = km_centers_fix
print(f'\n→ K yang dipakai untuk DEC: {N_CLUSTERS} (K fixed museum)')

np.save(os.path.join(CONFIG['output_root'], 'kmeans_labels.npy'), kmeans_labels)


## Cell 12 — Implementasi Deep Embedded Clustering (DEC)

In [ ]:
# ============================================================
# CELL 12: IMPLEMENTASI DEC
# Paper: Xie et al. 2016 — 'Unsupervised Deep Embedding
#         for Clustering Analysis'
# ============================================================

class ClusteringLayer(layers.Layer):
    """
    Custom layer: soft assignment q_ij via Student's t-distribution.
    q_ij = (1 + ||z_i - u_j||^2/alpha)^(-(alpha+1)/2)
           / sum_j'(1 + ||z_i - u_j'||^2/alpha)^(-(alpha+1)/2)
    """
    def __init__(self, n_clusters, weights=None, alpha=1.0, **kwargs):
        super().__init__(**kwargs)
        self.n_clusters   = n_clusters
        self.alpha        = alpha
        self.init_weights = weights

    def build(self, input_shape):
        self.clusters = self.add_weight(
            shape=(self.n_clusters, input_shape[-1]),
            initializer='glorot_uniform',
            trainable=True, name='cluster_centers'
        )
        if self.init_weights is not None:
            self.set_weights([self.init_weights])
        super().build(input_shape)

    def call(self, inputs):
        diff     = tf.expand_dims(inputs, axis=1) - self.clusters  # (N, K, d)
        sq_dist  = tf.reduce_sum(tf.square(diff), axis=2)          # (N, K)
        num      = tf.pow(1.0 + sq_dist / self.alpha,
                          -(self.alpha + 1.0) / 2.0)
        q        = num / tf.reduce_sum(num, axis=1, keepdims=True)
        return q

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'n_clusters': self.n_clusters, 'alpha': self.alpha})
        return cfg


def target_distribution(q):
    """
    Hitung distribusi target P dari soft assignment Q.
    p_ij = (q_ij^2 / f_j) / sum_j'(q_ij'^2 / f_j')
    Membuat distribusi lebih 'tajam' (meningkatkan confidence).
    """
    weight = q ** 2 / q.sum(axis=0)
    return (weight.T / weight.sum(axis=1)).T


def build_dec_model(encoder, n_clusters, init_centers):
    dec_input   = layers.Input(shape=encoder.input_shape[1:], name='dec_input')
    encoded     = encoder(dec_input)
    cluster_out = ClusteringLayer(
        n_clusters, weights=init_centers, name='clustering'
    )(encoded)
    return models.Model(inputs=dec_input, outputs=cluster_out, name='DEC')


dec_model = build_dec_model(encoder, N_CLUSTERS, kmeans_centers)
dec_model.compile(
    optimizer=Adam(learning_rate=CONFIG['dec_lr']),
    loss='kld'
)
print('=== Arsitektur DEC ===')
dec_model.summary()
print(f'\nCentroid diinisialisasi dari K-Means ({N_CLUSTERS} cluster, n_init=50)')


## Cell 13 — Training DEC (Curriculum Learning + ReduceLR)
> Anti-collapse strategy: monitor delta label, ReduceLROnPlateau jika konvergen lambat.

In [ ]:
# ============================================================
# CELL 13: TRAINING DEC — CURRICULUM LEARNING
# Stopping: delta_label < tol (proporsi gambar yang berganti label)
# Anti-collapse: cek distribusi cluster tidak kosong
# ============================================================

print('Mulai training DEC...')
print(f'  K (cluster)    : {N_CLUSTERS}')
print(f'  Latent Dim     : {LATENT_DIM}')
print(f'  Epochs max     : {CONFIG["dec_epochs"]}')
print(f'  Toleransi stop : {CONFIG["tol"]}')
print('-' * 55)

loss_history_dec = []
delta_label_hist = []
sil_hist         = []   # Silhouette per-epoch (setiap 10 epoch)
label_prev       = None
best_dec_loss    = np.inf
best_dec_labels  = None
best_sil         = -1.0
best_dec_labels_sil = None

n_samples    = images.shape[0]
batch_size   = CONFIG['dec_batch_size']

# Scheduler sederhana untuk Curriculum Learning
# Fase 1 (epoch 1-50): LR normal
# Fase 2 (epoch 51+) : LR dikurangi 0.5 (jika delta masih besar)
PHASE2_EPOCH = 50

for epoch in range(1, CONFIG['dec_epochs'] + 1):

    q = dec_model.predict(images, batch_size=batch_size, verbose=0)
    p = target_distribution(q)
    label_curr = q.argmax(axis=1)

    # Hitung delta label
    if label_prev is not None:
        delta = np.sum(label_curr != label_prev) / n_samples
    else:
        delta = 1.0
    delta_label_hist.append(delta)

    # Stopping criterion
    if epoch > 1 and delta < CONFIG['tol']:
        print(f'[Epoch {epoch:3d}] STOP: delta={delta:.5f} < tol={CONFIG["tol"]}')
        break

    # Anti-collapse check
    cluster_counts = np.bincount(label_curr, minlength=N_CLUSTERS)
    n_empty = np.sum(cluster_counts == 0)
    if n_empty > 0:
        print(f'[Epoch {epoch:3d}] WARNING: {n_empty} cluster kosong! Reinit dari K-Means...')
        km_re = KMeans(n_clusters=N_CLUSTERS, n_init=20, random_state=epoch)
        km_re.fit(latent_features)
        dec_model.get_layer('clustering').set_weights([km_re.cluster_centers_])
        label_prev = None
        continue

    label_prev = label_curr.copy()

    # Curriculum LR
    if epoch == PHASE2_EPOCH:
        new_lr = CONFIG['dec_lr'] * 0.5
        dec_model.optimizer.learning_rate.assign(new_lr)
        print(f'[Epoch {epoch:3d}] Fase 2: LR diturunkan ke {new_lr:.2e}')

    # Training satu epoch
    hist = dec_model.fit(images, p, batch_size=batch_size, epochs=1, verbose=0)
    epoch_loss = hist.history['loss'][0]
    loss_history_dec.append(epoch_loss)

    # Simpan model terbaik (loss)
    if epoch_loss < best_dec_loss:
        best_dec_loss   = epoch_loss
        best_dec_labels = label_curr.copy()
        dec_model.save(os.path.join(CONFIG['output_root'], 'model', 'dec_best_loss.keras'))

    # Silhouette per 10 epoch — simpan model terbaik Silhouette
    if epoch % 10 == 0:
        sil_epoch = silhouette_score(latent_features, label_curr)
        sil_hist.append((epoch, sil_epoch))
        if sil_epoch > best_sil:
            best_sil = sil_epoch
            best_dec_labels_sil = label_curr.copy()
            dec_model.save(os.path.join(CONFIG['output_root'], 'model', 'dec_best_sil.keras'))
        dist_str = ', '.join([f'C{u}:{c}' for u, c in
                              zip(*np.unique(label_curr, return_counts=True))])
        print(f'[Epoch {epoch:3d}] Loss={epoch_loss:.5f} | '
              f'Delta={delta:.4f} | Sil={sil_epoch:.4f} | {dist_str}')

# Label final (pilih dari model Silhouette terbaik)
q_final    = dec_model.predict(images, batch_size=batch_size, verbose=0)
dec_labels = q_final.argmax(axis=1)

# Hitung Silhouette final
dec_sil_final = silhouette_score(latent_features, dec_labels)
print(f'\n[FINAL] Silhouette DEC           : {dec_sil_final:.4f}')
print(f'[BEST ] Silhouette DEC (best ckpt): {best_sil:.4f}')

np.save(os.path.join(CONFIG['output_root'], 'dec_labels.npy'), dec_labels)
np.save(os.path.join(CONFIG['output_root'], 'q_final.npy'), q_final)
print('Label DEC tersimpan.')


## Cell 14 — Grafik Training DEC

In [ ]:
# ============================================================
# CELL 14: GRAFIK TRAINING DEC
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# KL Loss
axes[0].plot(range(1, len(loss_history_dec)+1), loss_history_dec,
             'b-o', ms=3, lw=2, label='KL Loss')
axes[0].set_title('KL-Divergence Loss (DEC)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('KL Loss')
axes[0].grid(True, linestyle='--', alpha=0.5)

# Delta label history
axes[1].plot(range(1, len(delta_label_hist)+1), delta_label_hist,
             'r-s', ms=3, lw=2, label='Delta Label')
axes[1].axhline(CONFIG['tol'], color='gray', linestyle='--', label=f'Tol={CONFIG["tol"]}')
axes[1].set_title('Proporsi Label Berubah per Epoch', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Delta Label')
axes[1].legend(); axes[1].grid(True, linestyle='--', alpha=0.5)

# Silhouette per 10 epoch
if sil_hist:
    sil_epochs, sil_vals = zip(*sil_hist)
    axes[2].plot(sil_epochs, sil_vals, 'g-D', ms=5, lw=2, label='Silhouette')
    axes[2].axhline(best_sil, color='darkgreen', linestyle='--', lw=1.5,
                    label=f'Best={best_sil:.4f}')
    axes[2].set_title('Silhouette Score per 10 Epoch', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Silhouette Score')
    axes[2].legend(); axes[2].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('Training DEC (Curriculum Learning)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '04_dec_training.png'),
            dpi=200, bbox_inches='tight')
plt.show()


## Cell 15 — Evaluasi Silhouette Komprehensif (per-sampel + overall)

In [ ]:
# ============================================================
# CELL 15: EVALUASI SILHOUETTE KOMPREHENSIF
# - Overall Silhouette, DBI, CHI untuk K-Means & DEC
# - Per-sample Silhouette (mendeteksi gambar yang salah cluster)
# ============================================================

def evaluate_clustering_full(features, labels, method_name):
    n_unique = len(np.unique(labels))
    if n_unique < 2:
        print(f'[ERROR] {method_name}: hanya {n_unique} cluster — evaluasi tidak bisa dilakukan')
        return None
    sil     = silhouette_score(features, labels)
    sil_smp = silhouette_samples(features, labels)
    dbi     = davies_bouldin_score(features, labels)
    chi     = calinski_harabasz_score(features, labels)
    print(f'\n=== {method_name} (K={len(np.unique(labels))}) ===')
    print(f'  Silhouette Score (overall) : {sil:.4f}  (optimal → 1.0)')
    print(f'  Davies-Bouldin Index       : {dbi:.4f}  (optimal → 0.0)')
    print(f'  Calinski-Harabasz Index    : {chi:.2f}  (semakin besar semakin baik)')
    print(f'  Silhouette per-cluster:')
    for cl in np.unique(labels):
        mask = labels == cl
        sil_cl = sil_smp[mask].mean()
        n_neg  = np.sum(sil_smp[mask] < 0)
        print(f'    Cluster {cl:2d}: mean={sil_cl:.4f}, negative={n_neg}/{mask.sum()}')
    return {'method': method_name, 'silhouette': float(round(sil, 4)),
            'davies_bouldin': float(round(dbi, 4)),
            'calinski_harabasz': float(round(chi, 2)),
            'silhouette_samples': sil_smp.tolist()}


result_km_fix = evaluate_clustering_full(latent_features, kmeans_labels, 'K-Means (K=10)')
result_dec    = evaluate_clustering_full(latent_features, dec_labels,    'DEC (K=10)')

# Plot: Silhouette per-sampel (DEC)
sil_smp_dec = np.array(result_dec['silhouette_samples'])
n_clust = len(np.unique(dec_labels))
cmap    = plt.cm.tab10

fig, ax = plt.subplots(figsize=(12, 6))
y_lower = 10
for cl in range(n_clust):
    cl_sil  = np.sort(sil_smp_dec[dec_labels == cl])
    size_cl = len(cl_sil)
    y_upper = y_lower + size_cl
    color   = cmap(cl / n_clust)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cl_sil,
                     facecolor=color, edgecolor=color, alpha=0.8)
    ax.text(-0.05, y_lower + 0.5 * size_cl, str(cl), fontsize=8, color=color)
    y_lower = y_upper + 10

ax.axvline(result_dec['silhouette'], color='red', linestyle='--', lw=2,
           label=f'Overall Sil = {result_dec["silhouette"]:.4f}')
ax.set_xlabel('Silhouette Coefficient', fontsize=12)
ax.set_ylabel('Cluster', fontsize=12)
ax.set_title('Silhouette Plot per Sampel — DEC', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '05_silhouette_plot_dec.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Simpan hasil evaluasi
eval_results = [result_km_fix, result_dec]
with open(os.path.join(CONFIG['output_root'], 'evaluasi_clustering.json'), 'w') as f:
    r_save = [{'method': r['method'], 'silhouette': r['silhouette'],
               'davies_bouldin': r['davies_bouldin'],
               'calinski_harabasz': r['calinski_harabasz']} for r in eval_results]
    json.dump(r_save, f, indent=2)


## Cell 16 — Visualisasi t-SNE (Sebelum vs Sesudah DEC)

In [ ]:
# ============================================================
# CELL 16: VISUALISASI t-SNE
# Membandingkan sebaran: K-Means vs DEC vs Ground Truth
# ============================================================

print('Menjalankan t-SNE (mungkin beberapa menit)...')
n_pca_tsne = min(50, latent_features.shape[1])
pca_pre    = PCA(n_components=n_pca_tsne, random_state=SEED)
feat_pca50 = pca_pre.fit_transform(latent_features)

tsne = TSNE(n_components=2, perplexity=CONFIG['tsne_perplexity'],
            n_iter=1000, random_state=SEED, verbose=1)
feat_2d = tsne.fit_transform(feat_pca50)
np.save(os.path.join(CONFIG['output_root'], 'tsne_2d.npy'), feat_2d)
print(f't-SNE selesai. Shape: {feat_2d.shape}')

# Plot 3 panel: K-Means | DEC | Ground Truth
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
n_k = len(np.unique(kmeans_labels))
n_d = len(np.unique(dec_labels))
n_g = len(class_names)
configs_tsne = [
    (kmeans_labels, n_k,  'K-Means (K=10)',          'tab10'),
    (dec_labels,    n_d,  'DEC (K=10)',               'tab10'),
    (gt_labels,     n_g,  'Ground Truth (10 folder)', 'tab10'),
]

for ax, (labels_t, n_t, title_t, cmap_t) in zip(axes, configs_tsne):
    cmap_fn = plt.cm.get_cmap(cmap_t, n_t)
    for cl in range(n_t):
        mask = labels_t == cl
        lbl  = class_names[cl] if (title_t.startswith('Ground') and cl < len(class_names)) else f'C{cl}'
        ax.scatter(feat_2d[mask, 0], feat_2d[mask, 1],
                   s=20, alpha=0.7, color=cmap_fn(cl), label=f'{lbl} ({mask.sum()})')
    ax.set_title(title_t, fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE 1'); ax.set_ylabel('t-SNE 2')
    ax.legend(fontsize=6, ncol=2, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.3)

plt.suptitle('Visualisasi t-SNE: K-Means | DEC | Ground Truth',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '06_tsne.png'),
            dpi=200, bbox_inches='tight')
plt.show()


## Cell 17 — Visualisasi PCA + Scree Plot

In [ ]:
# ============================================================
# CELL 17: PCA VISUALIZATION + SCREE PLOT
# ============================================================

pca_2d   = PCA(n_components=2, random_state=SEED)
feat_pca = pca_2d.fit_transform(latent_features)
var_exp  = pca_2d.explained_variance_ratio_

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

for ax, (lbl_t, n_t, title_t) in zip(axes[:2], [
    (dec_labels, len(np.unique(dec_labels)), 'DEC'),
    (gt_labels, len(class_names), 'Ground Truth'),
]):
    cmap_fn = plt.cm.get_cmap('tab10', n_t)
    for cl in range(n_t):
        mask = lbl_t == cl
        label_str = class_names[cl] if title_t == 'Ground Truth' else f'C{cl}'
        ax.scatter(feat_pca[mask, 0], feat_pca[mask, 1], s=20, alpha=0.7,
                   color=cmap_fn(cl), label=f'{label_str} ({mask.sum()})')
    ax.set_title(f'PCA 2D — {title_t}', fontsize=12, fontweight='bold')
    ax.set_xlabel(f'PC1 ({var_exp[0]*100:.1f}%)')
    ax.set_ylabel(f'PC2 ({var_exp[1]*100:.1f}%)')
    ax.legend(fontsize=6, ncol=2)
    ax.grid(True, linestyle='--', alpha=0.3)

# Scree plot
ax = axes[2]
pca_full = PCA(random_state=SEED).fit(latent_features)
cumvar   = np.cumsum(pca_full.explained_variance_ratio_)
ax.plot(range(1, len(cumvar)+1), cumvar * 100, 'b-o', ms=4, lw=2)
ax.axhline(90, color='red', linestyle='--', label='90% variance')
ax.axhline(95, color='orange', linestyle='--', label='95% variance')
n_90 = int(np.argmax(cumvar >= 0.90)) + 1
n_95 = int(np.argmax(cumvar >= 0.95)) + 1
ax.axvline(n_90, color='red', linestyle=':', alpha=0.7)
ax.axvline(n_95, color='orange', linestyle=':', alpha=0.7)
ax.text(n_90 + 0.5, 50, f'{n_90} dim\n→ 90%', fontsize=9, color='red')
ax.text(n_95 + 0.5, 35, f'{n_95} dim\n→ 95%', fontsize=9, color='orange')
ax.set_title('Scree Plot — Cumulative Variance\nLatent Space', fontsize=12, fontweight='bold')
ax.set_xlabel('Jumlah Komponen PCA')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Analisis PCA Latent Space', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '07_pca.png'),
            dpi=200, bbox_inches='tight')
plt.show()


## Cell 18 — Confusion Matrix: DEC vs Ground Truth Folder
> Menghitung seberapa cocok label DEC dengan kategori museum asli.
> Karena label DEC tidak memiliki urutan yang pasti, kita gunakan Hungarian algorithm
> untuk memetakan cluster DEC ke kategori ground truth secara optimal.

In [ ]:
# ============================================================
# CELL 18: CONFUSION MATRIX — DEC vs GROUND TRUTH FOLDER
# Hungarian Algorithm untuk mapping label optimal
# ============================================================

from scipy.optimize import linear_sum_assignment

def cluster_accuracy_hungarian(gt, pred, n_classes):
    """
    Menghitung akurasi clustering menggunakan Hungarian Algorithm
    untuk menemukan mapping terbaik antara label cluster dan ground truth.
    Returns: accuracy, best_mapping (dict cluster→gt)
    """
    n_pred = len(np.unique(pred))
    # Buat cost matrix (n_pred x n_classes)
    cost = np.zeros((n_pred, n_classes), dtype=int)
    for i in range(n_pred):
        for j in range(n_classes):
            cost[i, j] = np.sum((pred == i) & (gt == j))
    # Hungarian: minimisasi → kita negate agar maksimisasi overlap
    row_ind, col_ind = linear_sum_assignment(-cost)
    mapping = {row_ind[i]: col_ind[i] for i in range(len(row_ind))}
    # Terapkan mapping ke pred
    pred_mapped = np.array([mapping.get(p, -1) for p in pred])
    valid_mask  = pred_mapped != -1
    acc = np.sum(pred_mapped[valid_mask] == gt[valid_mask]) / len(gt)
    return acc, mapping, pred_mapped


n_classes  = len(class_names)

# Akurasi DEC
acc_dec, map_dec, dec_mapped = cluster_accuracy_hungarian(
    gt_labels, dec_labels, n_classes)

# Akurasi K-Means
acc_km, map_km, km_mapped = cluster_accuracy_hungarian(
    gt_labels, kmeans_labels, n_classes)

print(f'Cluster Accuracy (Hungarian Matching):')
print(f'  K-Means : {acc_km*100:.2f}%')
print(f'  DEC     : {acc_dec*100:.2f}%')

print(f'\nMapping DEC cluster → Ground Truth:')
for cl, gt_idx in sorted(map_dec.items()):
    n_matched = np.sum((dec_labels == cl) & (gt_labels == gt_idx))
    n_cl = np.sum(dec_labels == cl)
    print(f'  DEC C{cl:2d} → {class_names[gt_idx]:<30} ({n_matched}/{n_cl} cocok)')

# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, (pred_m, title_cm) in zip(axes, [
    (km_mapped, f'K-Means (Acc={acc_km*100:.1f}%)'),
    (dec_mapped, f'DEC (Acc={acc_dec*100:.1f}%)'),
]):
    valid = pred_m != -1
    cm = confusion_matrix(gt_labels[valid], pred_m[valid], labels=list(range(n_classes)))
    # Normalisasi per baris (recall)
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                ax=ax, linewidths=0.3, cbar=True)
    ax.set_title(title_cm, fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted (cluster mapped)', fontsize=10)
    ax.set_ylabel('Ground Truth (folder)', fontsize=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

plt.suptitle('Confusion Matrix: Cluster vs Ground Truth Folder Museum',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '08_confusion_matrix.png'),
            dpi=200, bbox_inches='tight')
plt.show()

# Simpan akurasi
accuracy_results = {
    'kmeans_cluster_accuracy': float(round(acc_km, 4)),
    'dec_cluster_accuracy':    float(round(acc_dec, 4)),
    'dec_mapping': {str(k): int(v) for k, v in map_dec.items()},
    'class_names': class_names
}
with open(os.path.join(CONFIG['output_root'], 'accuracy_results.json'), 'w') as f:
    json.dump(accuracy_results, f, indent=2)


## Cell 19 — Contoh Citra per Cluster (DEC & K-Means)

In [ ]:
# ============================================================
# CELL 19: VISUALISASI CONTOH CITRA PER CLUSTER
# ============================================================

def plot_cluster_samples(images, labels, class_names_mapping, n_clusters,
                          n_sample=6, title='DEC', save_path=None):
    """
    Grid: baris=cluster, kolom=sampel gambar
    class_names_mapping: dict cluster_id → nama kategori (dari Hungarian mapping)
    """
    fig = plt.figure(figsize=(n_sample * 2.0, n_clusters * 2.0))
    gs  = gridspec.GridSpec(n_clusters, n_sample, figure=fig,
                            hspace=0.4, wspace=0.1)
    cmap_fn = plt.cm.get_cmap('tab10', n_clusters)

    for cl in range(n_clusters):
        idx_cl = np.where(labels == cl)[0]
        if len(idx_cl) == 0:
            continue
        chosen = np.random.choice(idx_cl, min(n_sample, len(idx_cl)), replace=False)
        cat_name = class_names_mapping.get(cl, f'C{cl}')
        for j, img_idx in enumerate(chosen):
            ax = fig.add_subplot(gs[cl, j])
            ax.imshow(np.clip(images[img_idx], 0, 1))
            ax.axis('off')
            if j == 0:
                ax.set_ylabel(f'C{cl}\n({len(idx_cl)})\n{cat_name}',
                              fontsize=7, rotation=0, labelpad=55, va='center',
                              color=cmap_fn(cl))

    fig.suptitle(f'Contoh Citra per Cluster — {title}', fontsize=13,
                 fontweight='bold', y=1.01)
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Mapping cluster → nama kategori (dari Hungarian matching)
dec_cluster_names = {
    cl: class_names[gt_idx] for cl, gt_idx in map_dec.items()
}
km_cluster_names = {
    cl: class_names[gt_idx] for cl, gt_idx in map_km.items()
}

plot_cluster_samples(
    images, dec_labels, dec_cluster_names, N_CLUSTERS,
    n_sample=CONFIG['n_sample_vis'], title='DEC',
    save_path=os.path.join(CONFIG['output_root'], 'grafik', '09_citra_cluster_dec.png')
)

plot_cluster_samples(
    images, kmeans_labels, km_cluster_names, N_CLUSTERS,
    n_sample=CONFIG['n_sample_vis'], title='K-Means',
    save_path=os.path.join(CONFIG['output_root'], 'grafik', '10_citra_cluster_kmeans.png')
)


## Cell 20 — Simpan Gambar ke Folder Cluster + Manifest CSV

In [ ]:
# ============================================================
# CELL 20: SIMPAN GAMBAR KE FOLDER CLUSTER + CSV MANIFEST
# Struktur:
#   hasil_optimized/
#     cluster_dec/
#       cluster_00__arkeologi/
#       cluster_01__biologi/
#       ...
#     cluster_kmeans/
#       cluster_00__X/
# ============================================================

import csv

def save_clusters_to_folders(image_paths, labels, n_clusters,
                              output_dir, method_name='dec',
                              cluster_names=None):
    """
    Salin gambar ke folder cluster dengan nama kategori (jika tersedia).
    """
    base_dir = os.path.join(output_dir, f'cluster_{method_name}')
    for cl in range(n_clusters):
        name = cluster_names.get(cl, f'unknown') if cluster_names else 'unknown'
        # Bersihkan karakter spesial untuk nama folder
        safe_name = name.replace(' ', '_').replace('/', '_')
        os.makedirs(os.path.join(base_dir, f'cluster_{cl:02d}__{safe_name}'), exist_ok=True)

    manifest = []
    for path, label in tqdm(zip(image_paths, labels), total=len(image_paths),
                             desc=f'Simpan ({method_name})'):
        fname    = os.path.basename(path)
        name     = cluster_names.get(int(label), 'unknown') if cluster_names else 'unknown'
        safe_name = name.replace(' ', '_').replace('/', '_')
        dest_dir = os.path.join(base_dir, f'cluster_{label:02d}__{safe_name}')
        dest     = os.path.join(dest_dir, fname)
        if os.path.exists(dest):
            base, ext = os.path.splitext(fname)
            dest = os.path.join(dest_dir, f'{base}_{np.random.randint(10000)}{ext}')
        shutil.copy2(path, dest)
        manifest.append({
            'original_path'  : path,
            'cluster_id'     : int(label),
            'cluster_name'   : name,
            'gt_folder'      : os.path.basename(os.path.dirname(path)),
            'saved_to'       : dest
        })

    # Tulis CSV
    manifest_path = os.path.join(output_dir, f'manifest_{method_name}.csv')
    with open(manifest_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(manifest[0].keys()))
        writer.writeheader()
        writer.writerows(manifest)

    print(f'[{method_name.upper()}] Selesai → {base_dir}')
    for cl in range(n_clusters):
        n = sum(1 for m in manifest if m['cluster_id'] == cl)
        cname = cluster_names.get(cl, '?') if cluster_names else '?'
        print(f'  cluster_{cl:02d}__{cname}: {n} gambar')
    return manifest


manifest_dec = save_clusters_to_folders(
    image_paths, dec_labels, N_CLUSTERS,
    CONFIG['output_root'], method_name='dec',
    cluster_names=dec_cluster_names
)

manifest_km = save_clusters_to_folders(
    image_paths, kmeans_labels, N_CLUSTERS,
    CONFIG['output_root'], method_name='kmeans',
    cluster_names=km_cluster_names
)


## Cell 21 — Ringkasan Akhir & Skor Komparatif

In [ ]:
# ============================================================
# CELL 21: RINGKASAN AKHIR — KOMPARASI LENGKAP
# ============================================================

print('\n' + '='*65)
print(' RINGKASAN HASIL — DEC CLUSTERING OPTIMIZED')
print('='*65)
print(f'Dataset              : {len(images)} gambar ({len(class_names)} kategori museum)')
print(f'Latent Dim Optimal   : {LATENT_DIM} (dari grid search {CONFIG["latent_search"]})')
print(f'K Optimal (Elbow)    : {K_OPTIMAL}')
print(f'K yang dipakai (DEC) : {N_CLUSTERS} (fixed museum)')
print(f'AE Epochs (aktual)   : {len(history_ae.history["loss"])}')
print(f'DEC Epochs (aktual)  : {len(loss_history_dec)}')
print()
print(f'{"Metrik":<30} {"K-Means":>12} {"DEC":>12}')
print('-'*56)
print(f'{"Silhouette Score":<30} {result_km_fix["silhouette"]:>12.4f} {result_dec["silhouette"]:>12.4f}')
print(f'{"Davies-Bouldin Index":<30} {result_km_fix["davies_bouldin"]:>12.4f} {result_dec["davies_bouldin"]:>12.4f}')
print(f'{"Calinski-Harabasz":<30} {result_km_fix["calinski_harabasz"]:>12.2f} {result_dec["calinski_harabasz"]:>12.2f}')
print(f'{"Cluster Accuracy (Hungarian)":<30} {acc_km*100:>11.2f}% {acc_dec*100:>11.2f}%')
print('='*56)
print(f'Semua output tersimpan di: {CONFIG["output_root"]}')
print('='*65)

# Simpan ringkasan
summary = {
    'dataset'           : {'n_images': len(images), 'n_classes': len(class_names),
                           'class_names': class_names},
    'config'            : {'latent_dim': LATENT_DIM, 'k_optimal': K_OPTIMAL,
                           'n_clusters': N_CLUSTERS},
    'ae_epochs_run'     : len(history_ae.history['loss']),
    'dec_epochs_run'    : len(loss_history_dec),
    'kmeans': {'silhouette': result_km_fix['silhouette'],
               'davies_bouldin': result_km_fix['davies_bouldin'],
               'calinski_harabasz': result_km_fix['calinski_harabasz'],
               'cluster_accuracy': float(round(acc_km, 4))},
    'dec'   : {'silhouette': result_dec['silhouette'],
               'davies_bouldin': result_dec['davies_bouldin'],
               'calinski_harabasz': result_dec['calinski_harabasz'],
               'cluster_accuracy': float(round(acc_dec, 4))},
    'k_search': k_search_results,
    'latent_dim_search': ld_results,
}
with open(os.path.join(CONFIG['output_root'], 'ringkasan_final.json'), 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Bar chart komparatif
metrics = ['Silhouette\nScore', 'Cluster Accuracy\n(Hungarian)']
km_vals = [result_km_fix['silhouette'], acc_km]
dc_vals = [result_dec['silhouette'],    acc_dec]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(metrics))
w = 0.35
b1 = ax.bar(x - w/2, km_vals, w, label='K-Means', color='steelblue', edgecolor='white')
b2 = ax.bar(x + w/2, dc_vals, w, label='DEC',     color='darkorange', edgecolor='white')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.set_title('Komparasi K-Means vs DEC — Optimized Pipeline',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(True, axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, max(max(km_vals), max(dc_vals)) * 1.2)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_root'], 'grafik', '11_komparasi_final.png'),
            dpi=200, bbox_inches='tight')
plt.show()

print('\nNotebook DEC_Optimized selesai dieksekusi.')
print(f'Lihat hasil di: {CONFIG["output_root"]}')
